# Chương 6 — Học sâu trên đồ thị: GCN, GraphSAGE, GAT tự cài đặt trên PubMed-Diabetes, Cora và CiteSeer

In [1]:
# GNN-00 · Thiết lập đường dẫn: mã chương dùng chung gói của tiểu luận (src/dm) và mã kế thừa của nhóm (src/tl01, tl03)
import os, sys, json, time, warnings
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
os.environ["CUDA_VISIBLE_DEVICES"] = ""
from pathlib import Path
warnings.filterwarnings("ignore")
TLF = Path.cwd().parent
for p in ["src", "src/tl01", "src/tl03"]:
    sys.path.insert(0, str(TLF / p))
import numpy as np, pandas as pd
np.set_printoptions(precision=4, suppress=True)
print("Thư mục tiểu luận:", TLF)

Thư mục tiểu luận: /home/ngtukien/Projects/MultiAgent/tieuluan_final


Notebook kiểm chứng phép chuẩn hóa ma trận kề thưa $\hat{A} = \tilde{D}^{-1/2}(A+I)\tilde{D}^{-1/2}$, biểu diễn GCN dưới dạng hợp các hàm, so sánh phân loại nút bán giám sát trên ba đồ thị trích dẫn, và khảo sát hiện tượng làm trơn quá mức (over-smoothing) khi tăng số lớp.

In [2]:
# GNN-01 · Đặc trưng cấu trúc của ba đồ thị trích dẫn: PubMed-Diabetes, Cora và CiteSeer
d_stats = json.loads((TLF / "src/tl03/outputs/metrics/ch6_data.json").read_text())
pd.DataFrame(d_stats).T.round(3)

,nodes,edges,features,classes,train,val,test,deg_mean,deg_max,deg_median,isolated,homophily,feat_nnz_mean,density,class_counts,class_names,deg_hist
PubMed,19717,44324,500,3,60,500,1000,4.496019,171,2.0,0,0.802387,50.110615,0.000228,"[4103, 7739, 7875]","[ĐTĐ thực nghiệm, ĐTĐ típ 1, ĐTĐ típ 2]","[0, 9094, 3357, 1584, 914, 642, 491, 423, 316,..."
Cora,2708,5278,1433,7,140,500,1000,3.89808,168,3.0,0,0.809966,18.174299,0.00144,"[351, 217, 418, 818, 426, 298, 180]","[Case_Based, Genetic_Algorithms, Neural_Networ...","[0, 485, 583, 553, 389, 281, 131, 82, 57, 25, ..."
CiteSeer,3327,4552,3703,6,120,500,1000,2.736399,99,2.0,48,0.735501,31.609558,0.000823,"[264, 590, 668, 701, 596, 508]","[Agents, AI, DB, IR, ML, HCI]","[48, 1331, 795, 438, 237, 139, 110, 61, 33, 24..."


In [3]:
# GNN-02 · Kiểm chứng lớp GCN tự cài đặt bằng nhân ma trận thưa và so sánh số tham số với MLP
import torch
from tl3.ch6_gnn import load_planetoid, gcn_norm_adj, GNN
data = load_planetoid("PubMed")
adj = gcn_norm_adj(data.edge_index, data.num_nodes)
gcn = GNN("gcn", data.num_features, 64, 3)
mlp = GNN("mlp", data.num_features, 64, 3)
print("Ma trận kề chuẩn hóa Â:", adj.shape, "số phần tử khác 0:", adj._nnz())
print("Số tham số GCN:", sum(p.numel() for p in gcn.parameters()), "| MLP:", sum(p.numel() for p in mlp.parameters()))

Ma trận kề chuẩn hóa Â: torch.Size([19717, 19717]) số phần tử khác 0: 108365
Số tham số GCN: 32259 | MLP: 32259


In [4]:
# GNN-03 · Kết quả phân loại nút bán giám sát (5 hạt giống) trên PubMed-Diabetes, Cora và CiteSeer
node_res = json.loads((TLF / "src/tl03/outputs/metrics/ch6_node.json").read_text())
rows = []
for ds, models in node_res.items():
    for m, v in models.items():
        rows.append({"đồ thị": ds, "mô hình": m, "test_acc": v["test_mean"], "std": v.get("test_std", 0.0),
                     "macro_f1": v.get("f1_mean", 0.0), "tham số": v["params"]})
pd.DataFrame(rows).pivot(index="mô hình", columns="đồ thị", values=["test_acc", "std"]).round(4)

test_acc                      std                
đồ thị  CiteSeer    Cora  PubMed CiteSeer    Cora  PubMed
mô hình                                                  
gat       0.7152  0.8290  0.7772   0.0077  0.0108  0.0035
gcn       0.7210  0.8212  0.7890   0.0087  0.0082  0.0048
logreg    0.6140  0.5740  0.7350   0.0000  0.0000  0.0000
mlp       0.5756  0.5846  0.7224   0.0066  0.0071  0.0045
sage      0.6988  0.8036  0.7650   0.0073  0.0059  0.0049

In [5]:
# GNN-04 · Khảo sát độ sâu (1–8 lớp) và hiện tượng làm trơn quá mức (MAD) khi có/không có kết nối tắt
depth_res = json.loads((TLF / "src/tl03/outputs/metrics/ch6_depth.json").read_text())["rows"]
pd.DataFrame(depth_res).round(4)

,layers,residual,test_mean,test_std,mad,mad_neighbors
0,1,False,0.7637,0.0082,0.4875,0.0848
1,2,False,0.8213,0.0083,0.1598,0.0412
2,3,False,0.8023,0.0026,0.3082,0.0257
3,3,True,0.8153,0.0046,0.2748,0.0256
4,4,False,0.7440,0.0259,0.3442,0.0240
5,4,True,0.7903,0.0070,0.3352,0.0255
6,5,False,0.6290,0.0892,0.3634,0.0288
7,5,True,0.7917,0.0155,0.3811,0.0267
8,6,False,0.5133,0.2009,0.2797,0.0287
9,6,True,0.7893,0.0070,0.3912,0.0284
